In [1]:
import pandas as pd
import numpy as np
import os
import json
import glob
from collections import defaultdict

os.makedirs("../data/clean", exist_ok=True)

In [2]:
# ============================================
# 1. Multi-year file (2013–2023)
# ============================================
df_main = pd.read_csv("../data/raw/auction/auction.csv")
df_main = df_main.drop(columns=["Unnamed: 0"])
df_main = df_main.rename(columns={
    "Player": "player_name", "Team": "team",
    "Base price": "base_price_lakh", "Winning bid": "price_paid_lakh",
    "Year": "year",
})

df_main["price_paid_lakh"] = (
    df_main["price_paid_lakh"].astype(str).str.replace(",", "").astype(float)
)
df_main["base_price_lakh"] = pd.to_numeric(df_main["base_price_lakh"], errors="coerce")

df_main["base_price_cr"] = df_main["base_price_lakh"] / 100
df_main["price_paid_cr"] = df_main["price_paid_lakh"] / 100
df_main["status"] = "sold"
df_main = df_main[["player_name", "year", "team", "base_price_cr", "price_paid_cr", "status"]]

print(df_main.shape)
df_main.head()

(1052, 6)


,player_name,year,team,base_price_cr,price_paid_cr,status
0,Christopher Barnwell,2013,Royal Challengers Bangalore,0.305,0.305,sold
1,Johan Botha,2013,Delhi Daredevils,1.830,2.745,sold
2,Daniel Christian,2013,Royal Challengers Bangalore,0.610,0.610,sold
3,Michael Clarke,2013,Pune Warriors India,2.440,2.440,sold
4,Nathan Coulter-Nile,2013,Mumbai Indians,0.610,2.745,sold


In [3]:
# ============================================
# 2. 2025 file — split into sold / unsold / drop TBA
# ============================================
df_2025 = pd.read_csv("../data/raw/auction/auction_2025.csv")
df_2025 = df_2025.rename(columns={
    "Players": "player_name", "Team": "team",
    "Type": "role", "Base": "base_price_cr", "Sold": "price_paid_cr",
})

df_2025 = df_2025[df_2025["price_paid_cr"] != "TBA"].copy()
df_2025["base_price_cr"] = df_2025["base_price_cr"].replace("-", np.nan).astype(float)
df_2025["status"] = np.where(df_2025["price_paid_cr"] == "Unsold", "unsold", "sold")
df_2025["price_paid_cr"] = pd.to_numeric(
    df_2025["price_paid_cr"].replace("Unsold", np.nan), errors="coerce"
)
df_2025["year"] = 2025
df_2025 = df_2025[["player_name", "year", "team", "base_price_cr", "price_paid_cr", "status"]]

print(df_2025.shape)
df_2025.head()

(330, 6)


,player_name,year,team,base_price_cr,price_paid_cr,status
0,Virat Kohli,2025,RCB,NaN,21.00,sold
1,Rajat Patidar,2025,RCB,NaN,11.00,sold
2,Yash Dayal,2025,RCB,NaN,5.00,sold
3,Jasprit Bumrah,2025,MI,NaN,18.00,sold
4,Suryakumar Yadav,2025,MI,NaN,16.35,sold


In [4]:
# ============================================
# 3. Combine sold records from both files
# ============================================
auction_history = pd.concat([
    df_main,
    df_2025[df_2025["status"] == "sold"]
], ignore_index=True)
auction_history.to_csv("../data/clean/auction_history.csv", index=False)

print(auction_history.shape)

(1279, 6)


In [5]:
# ============================================
# 4. Separate unsold table (2025 only, for now)
# ============================================
unsold_2025 = df_2025[df_2025["status"] == "unsold"]
unsold_2025.to_csv("../data/clean/unsold_2025.csv", index=False)

print(unsold_2025.shape)

(103, 6)


In [6]:
# ============================================
# 5. Sanity check
# ============================================
print("auction_history shape:", auction_history.shape)
print(auction_history.groupby("year").size())
print()
print("unsold_2025 shape:", unsold_2025.shape)
print(auction_history.head())

auction_history shape: (1279, 6)
year
2013     37
2014    154
2015     66
2016     94
2017     66
2018    169
2019     60
2020     62
2021     57
2022    204
2023     83
2025    227
dtype: int64

unsold_2025 shape: (103, 6)
            player_name  year                         team  base_price_cr  \
0  Christopher Barnwell  2013  Royal Challengers Bangalore          0.305   
1           Johan Botha  2013             Delhi Daredevils          1.830   
2      Daniel Christian  2013  Royal Challengers Bangalore          0.610   
3        Michael Clarke  2013          Pune Warriors India          2.440   
4   Nathan Coulter-Nile  2013               Mumbai Indians          0.610   

   price_paid_cr status  
0          0.305   sold  
1          2.745   sold  
2          0.610   sold  
3          2.440   sold  
4          2.745   sold  


In [7]:
# ============================================
# 6. Load Cricsheet IPL match files
# ============================================
files = glob.glob("../data/raw/cricsheet_ipl/*.json")
print(f"Total match files: {len(files)}")

with open(files[0]) as f:
    match = json.load(f)

print(match["info"]["teams"])
print(match["info"]["dates"])
print(match["info"].get("season", "no season field"))

Total match files: 1243
['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
['2017-04-05']
2017


In [8]:
# ============================================
# 7. Season coverage + normalization
# ============================================
seasons = []
for fp in files:
    with open(fp) as f:
        m = json.load(f)
    seasons.append(str(m["info"].get("season", "unknown")))

season_map = {"2007/08": "2008", "2009/10": "2010", "2020/21": "2020"}
seasons_clean = [season_map.get(s, s) for s in seasons]
print(pd.Series(seasons_clean).value_counts().sort_index())

2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74
2026    74
Name: count, dtype: int64


In [9]:
# ============================================
# 8. Extract unique player names from both sources
# ============================================
cricsheet_names = set()
for fp in files:
    with open(fp) as f:
        m = json.load(f)
    for team, players in m["info"].get("players", {}).items():
        cricsheet_names.update(players)

auction_names = set(auction_history["player_name"].unique())
print(f"Unique Cricsheet names: {len(cricsheet_names)}")
print(f"Unique auction names: {len(auction_names)}")

Unique Cricsheet names: 811
Unique auction names: 673


In [10]:
# ============================================
# 9. Player name matching (exact -> surname+initial -> ambiguous/unmatched)
# ============================================
def normalize_name(name):
    parts = str(name).strip().split()
    if not parts:
        return (None, None)
    return (parts[-1].lower(), parts[0][0].lower())

cricsheet_lookup = defaultdict(list)
for name in cricsheet_names:
    cricsheet_lookup[normalize_name(name)].append(name)

matched_rows = []
ambiguous_rows = []
unmatched_rows = []

for name in auction_names:
    if name in cricsheet_names:
        matched_rows.append({"auction_name": name, "cricsheet_name": name})
        continue

    key = normalize_name(name)
    candidates = cricsheet_lookup.get(key, [])

    if len(candidates) == 1:
        matched_rows.append({"auction_name": name, "cricsheet_name": candidates[0]})
    elif len(candidates) > 1:
        if name in candidates:
            matched_rows.append({"auction_name": name, "cricsheet_name": name})
        else:
            ambiguous_rows.append({"auction_name": name, "candidates": candidates})
    else:
        unmatched_rows.append({"auction_name": name})

print(f"Clean matches: {len(matched_rows)}")
print(f"Ambiguous: {len(ambiguous_rows)}")
print(f"No match: {len(unmatched_rows)}")

Clean matches: 518
Ambiguous: 34
No match: 121


In [11]:
# ============================================
# 10. Export ambiguous + unmatched for manual review
# ============================================
ambiguous_df = pd.DataFrame(ambiguous_rows)
ambiguous_df["candidates"] = ambiguous_df["candidates"].apply(lambda x: "; ".join(x))
ambiguous_df["chosen_cricsheet_name"] = ""
ambiguous_df["match_type"] = "ambiguous"

unmatched_df = pd.DataFrame(unmatched_rows)
unmatched_df["candidates"] = ""
unmatched_df["chosen_cricsheet_name"] = ""
unmatched_df["match_type"] = "no_match"

review_df = pd.concat([ambiguous_df, unmatched_df], ignore_index=True)
review_df = review_df[["auction_name", "match_type", "candidates", "chosen_cricsheet_name"]]
review_df.to_csv("../data/clean/unmatched_names.csv", index=False)

print(review_df.shape)
review_df.head(10)

(155, 4)


,auction_name,match_type,candidates,chosen_cricsheet_name
0,M. Shahrukh Khan,ambiguous,M Shahrukh Khan; Mohsin Khan; Musheer Khan,
1,Rahul Sharma,ambiguous,R Sharma; Raghu Sharma; RG Sharma,
2,Rinku Singh,ambiguous,RK Singh; Ravi Singh; RP Singh; Ramandeep Singh,
3,Dwayne Bravo,ambiguous,DM Bravo; DJ Bravo,
4,Mohammed Arshad Khan,ambiguous,M Shahrukh Khan; Mohsin Khan; Musheer Khan,
5,Fabian Allen,ambiguous,FH Allen; FA Allen,
6,Avinash Singh,ambiguous,Anureet Singh; Anmolpreet Singh; Anirudh Singh...,
7,Rohit Sharma,ambiguous,R Sharma; Raghu Sharma; RG Sharma,
8,Veer Pratap Singh,ambiguous,V Pratap Singh; VRV Singh; Virat Singh,
9,Saurabh Kumar,ambiguous,Sumit Kumar; Shivang Kumar,


In [ ]:
# ============================================
# 11. Show team/year context for each ambiguous candidate
# ============================================
def get_player_context(player_name, match_files):
    teams_seen = set()
    years_seen = set()
    for fp in match_files:
        with open(fp) as f:
            m = json.load(f)
        for team, players in m["info"].get("players", {}).items():
            if player_name in players:
                teams_seen.add(team)
                years_seen.add(str(m["info"].get("season", "unknown")))
    return sorted(teams_seen), sorted(years_seen)

for idx, row in ambiguous_df.iterrows():
    print(f"\n=== Auction name: {row['auction_name']} ===")
    for candidate in row["candidates"].split("; "):
        teams, years = get_player_context(candidate, files)
        print(f"  {candidate}: teams={teams}, years={years}")


=== Auction name: M. Shahrukh Khan ===
  M Shahrukh Khan: teams=['Gujarat Titans', 'Punjab Kings'], years=['2021', '2022', '2023', '2024', '2025', '2026']
  Mohsin Khan: teams=['Lucknow Super Giants'], years=['2022', '2023', '2024', '2026']
  Musheer Khan: teams=['Punjab Kings'], years=['2025']

=== Auction name: Rahul Sharma ===
  R Sharma: teams=['Deccan Chargers', 'Delhi Daredevils', 'Pune Warriors'], years=['2009/10', '2011', '2012', '2013', '2014']
  Raghu Sharma: teams=['Mumbai Indians'], years=['2026']
  RG Sharma: teams=['Deccan Chargers', 'Mumbai Indians'], years=['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

=== Auction name: Rinku Singh ===
  RK Singh: teams=['Kolkata Knight Riders'], years=['2018', '2019', '2020/21', '2022', '2023', '2024', '2025', '2026']
  Ravi Singh: teams=['Rajasthan Royals'], years=['2026']
  RP Singh: teams=['Deccan Chargers', 'Kochi Tu

In [ ]:
# ============================================
# 12. Apply manual resolutions to ambiguous names
# ============================================
manual_resolution = {
    "Dwayne Bravo": "DJ Bravo", "Darren Bravo": "DM Bravo",
    "Rinku Singh": "RK Singh", "Prabhsimran Singh": "P Simran Singh",
    "Prab Simran Singh": "P Simran Singh", "Aman Khan": "Aman Hakim Khan",
    "Kuldip Yadav": "Kuldeep Yadav", "Chris Green": "CJ Green",
    "Cameron Green": "C Green", "M. Shahrukh Khan": "M Shahrukh Khan",
    "Yalaka Venugopal Rao": "Y Venugopal Rao", "Rohit Sharma": "RG Sharma",
    "Veer Pratap Singh": "V Pratap Singh", "Shubham Dubey": "SB Dubey",
    "R. P. Singh": "RP Singh", "Praveen Kumar": "P Kumar",
    "Jitesh Sharma": "JM Sharma", "Ankit Nagendra Sharma": "Ankit Sharma",
    "Rahul Sharma": "R Sharma", "Karan Sharma": "KS Sharma",
    "Karn Sharma": "KV Sharma", "Fabian Allen": "FA Allen",
    "Finn Allen": "FH Allen",
}

for name, cricsheet_name in manual_resolution.items():
    matched_rows.append({"auction_name": name, "cricsheet_name": cricsheet_name})

still_unresolved = [row["auction_name"] for row in ambiguous_rows if row["auction_name"] not in manual_resolution]
print(f"Total matched now: {len(matched_rows)}")
print(f"Still unresolved (ambiguous, no confident pick): {len(still_unresolved)}")
print(still_unresolved)

Total matched now: 541
Still unresolved (ambiguous, no confident pick): 11
['Mohammed Arshad Khan', 'Pratyush Singh', 'Anunay Singh', 'Kshitiz Sharma', 'Harnoor Singh', 'Manvanth Kumar', 'Saurabh Dubey', 'Saurabh Kumar', 'Pratham Singh', 'Milind Kumar', 'Avinash Singh']


In [ ]:
# ============================================
# 13. Build the final player_id_map.csv
# ============================================
matched_df = pd.DataFrame(matched_rows).drop_duplicates(subset="auction_name").reset_index(drop=True)
matched_df["player_id"] = ["p" + str(i+1).zfill(4) for i in range(len(matched_df))]
matched_df = matched_df[["player_id", "auction_name", "cricsheet_name"]]

matched_df.to_csv("../data/clean/player_id_map.csv", index=False)

print(matched_df.shape)
matched_df.head(10)

(541, 3)


,player_id,auction_name,cricsheet_name
0,p0001,Rashid Khan,Rashid Khan
1,p0002,AM Ghazanfar,AM Ghazanfar
2,p0003,Moises Henriques,MC Henriques
3,p0004,Krishnappa Gowtham,K Gowtham
4,p0005,Ajinkya Rahane,AM Rahane
5,p0006,Mitchell McClenaghan,MJ McClenaghan
6,p0007,Virat Singh,Virat Singh
7,p0008,Odean Smith,OF Smith
8,p0009,Yash Dhull,YV Dhull
9,p0010,Vidwath Kaverappa,V Kaverappa


In [ ]:
# ============================================
# 14. Refresh unmatched_names.csv with final remaining gaps
# ============================================
remaining_ambiguous = [row for row in ambiguous_rows if row["auction_name"] not in manual_resolution]

ambiguous_df_final = pd.DataFrame(remaining_ambiguous)
ambiguous_df_final["candidates"] = ambiguous_df_final["candidates"].apply(lambda x: "; ".join(x))
ambiguous_df_final["chosen_cricsheet_name"] = ""
ambiguous_df_final["match_type"] = "ambiguous"

unmatched_df_final = pd.DataFrame(unmatched_rows)
unmatched_df_final["candidates"] = ""
unmatched_df_final["chosen_cricsheet_name"] = ""
unmatched_df_final["match_type"] = "no_match"

review_df_final = pd.concat([ambiguous_df_final, unmatched_df_final], ignore_index=True)
review_df_final = review_df_final[["auction_name", "match_type", "candidates", "chosen_cricsheet_name"]]
review_df_final.to_csv("../data/clean/unmatched_names.csv", index=False)

print(review_df_final.shape)

(132, 4)


In [ ]:
import os
os.makedirs("../data/raw/player_meta", exist_ok=True)

df_meta_2025 = pd.read_csv("../data/raw/player_meta/Player data.csv")
print(df_meta_2025.columns.tolist())
print(df_meta_2025.head())

['TATA IPL 2025 Auction List-15.12.24', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20']
  TATA IPL 2025 Auction List-15.12.24 Unnamed: 1 Unnamed: 2  Unnamed: 3  \
0                                 NaN        NaN        NaN         NaN   
1                                 NaN        NaN        NaN         NaN   
2                         List Sr.No.    Set No.   2025 Set  First Name   
3                                 NaN        NaN        NaN         NaN   
4                                 NaN        NaN        NaN         NaN   

  Unnamed: 4 Unnamed: 5   Unnamed: 6 Unnamed: 7 Unnamed: 8  Unnamed: 9  \
0        NaN        NaN          NaN        NaN        NaN         NaN   
1        NaN        NaN        State        NaN        NaN         NaN

In [ ]:
df_raw = pd.read_csv("../data/raw/player_meta/Player data.csv", header=None)
print(df_raw.head(6).T.to_string())

                                      0        1      2                    3            4     5
0   TATA IPL 2025 Auction List-15.12.24      NaN    NaN          List Sr.No.          NaN   NaN
1                                   NaN      NaN    NaN              Set No.          NaN   NaN
2                                   NaN      NaN    NaN             2025 Set          NaN   NaN
3                                   NaN      NaN    NaN           First Name          NaN   NaN
4                                   NaN      NaN    NaN              Surname          NaN   NaN
5                                   NaN      NaN    NaN              Country          NaN   NaN
6                                   NaN      NaN  State                  NaN  Association   NaN
7                                   NaN      NaN    NaN                  DOB          NaN   NaN
8                                   NaN      NaN    NaN                  Age          NaN   NaN
9                                   NaN 

In [ ]:
df_raw = pd.read_csv("../data/raw/player_meta/Player data.csv", header=None)

column_names = [
    "list_sr_no", "set_no", "set_2025", "first_name", "surname", "country",
    "state_association", "dob", "age", "specialism", "blank1", "blank2",
    "test_caps", "odi_caps", "t20_caps", "ipl_caps", "previous_ipl_teams",
    "team_2024", "ipl_2024", "c_u_a", "reserve_price_lakh"
]

df_meta_2025 = df_raw.iloc[6:].reset_index(drop=True)
df_meta_2025.columns = column_names[:df_meta_2025.shape[1]]

print(df_meta_2025.shape)
print(df_meta_2025.head(10))

(631, 21)
  list_sr_no set_no set_2025 first_name      surname       country  \
0          1      1       M1        Jos      Buttler       England   
1          2      1       M1    Shreyas         Iyer         India   
2          3      1       M1    Rishabh         Pant         India   
3          4      1       M1     Kagiso       Rabada  South Africa   
4          5      1       M1   Arshdeep        Singh         India   
5          6      1       M1   Mitchell        Starc     Australia   
6          7      2       M2  Yuzvendra       Chahal         India   
7          8      2       M2       Liam  Livingstone       England   
8          9      2       M2      David       Miller  South Africa   
9         10      2       M2         KL        Rahul         India   

  state_association         dob age    specialism blank1  \
0               NaN  08/09/1990  34  WICKETKEEPER    RHB   
1               MCA  06/12/1994  30        BATTER    RHB   
2              DDCA  04/10/1997  27    

In [ ]:
column_names = [
    "list_sr_no", "set_no", "set_2025", "first_name", "surname", "country",
    "state_association", "dob", "age", "specialism", "batting_style", "bowling_style",
    "test_caps", "odi_caps", "t20_caps", "ipl_caps", "previous_ipl_teams",
    "team_2024", "ipl_2024_note", "c_u_a", "reserve_price_lakh"
]

df_meta_2025 = pd.read_csv(
    "../data/raw/player_meta/Player data.csv",
    skiprows=6,
    header=None,
    names=column_names
)

print(df_meta_2025.shape)
df_meta_2025.head(10)

(631, 21)


,list_sr_no,set_no,set_2025,first_name,surname,country,state_association,dob,age,specialism,batting_style,bowling_style,test_caps,odi_caps,t20_caps,ipl_caps,previous_ipl_teams,team_2024,ipl_2024_note,c_u_a,reserve_price_lakh
0,1,1,M1,Jos,Buttler,England,NaN,08/09/1990,34,WICKETKEEPER,RHB,RIGHT ARM Off Spin,57,181,124,107,"MI,RR",RR,11,Capped,200
1,2,1,M1,Shreyas,Iyer,India,MCA,06/12/1994,30,BATTER,RHB,RIGHT ARM Off Spin,14,62,51,116,"DC,KKR",KKR,15,Capped,200
2,3,1,M1,Rishabh,Pant,India,DDCA,04/10/1997,27,BATTER,LHB,RIGHT ARM Off Spin,37,31,76,111,DC,DC,13,Capped,200
3,4,1,M1,Kagiso,Rabada,South Africa,NaN,25/05/1995,29,BOWLER,RHB,RIGHT ARM Fast,66,101,65,80,"DC,PBKS",PBKS,11,Capped,200
4,5,1,M1,Arshdeep,Singh,India,PCA,05/02/1999,26,BOWLER,LHB,LEFT ARM Fast Medium,NaN,8,56,65,PBKS,PBKS,14,Capped,200
5,6,1,M1,Mitchell,Starc,Australia,NaN,30/01/1990,35,BOWLER,LHB,LEFT ARM Fast,89,125,65,41,RCB.KKR,KKR,14,Capped,200
6,7,2,M2,Yuzvendra,Chahal,India,HCA,23/07/1990,34,BOWLER,RHB,RIGHT ARM Leg Spin,NaN,72,80,160,"MI,RCB,RR",RR,15,Capped,200
7,8,2,M2,Liam,Livingstone,England,NaN,04/08/1993,31,ALL-ROUNDER,RHB,RIGHT ARM Leg Spin,1,31,50,39,"RR,PBKS",PBKS,7,Capped,200
8,9,2,M2,David,Miller,South Africa,NaN,10/06/1989,35,BATTER,LHB,NaN,NaN,173,125,130,"PBKS,RR,GT",GT,9,Capped,150
9,10,2,M2,KL,Rahul,India,KSCA,18/04/1992,32,WICKETKEEPER,RHB,NaN,53,77,72,132,"SRH,RCB,PBKS,LSG",LSG,14,Capped,200


In [ ]:
df_meta_2025["full_name"] = (
    df_meta_2025["first_name"].astype(str) + " " + df_meta_2025["surname"].astype(str)
).str.strip()

# How many of these names match your existing auction_name list exactly?
player_id_map = pd.read_csv("../data/clean/player_id_map.csv")
meta_names = set(df_meta_2025["full_name"])
auction_id_names = set(player_id_map["auction_name"])

exact_overlap = meta_names & auction_id_names
print(f"Meta file has: {len(meta_names)} unique names")
print(f"Exact matches with player_id_map: {len(exact_overlap)}")
print(f"In meta but not matched: {len(meta_names - auction_id_names)}")

Meta file has: 574 unique names
Exact matches with player_id_map: 241
In meta but not matched: 333


In [ ]:
df_meta_2025 = df_meta_2025.rename(columns={
    "blank1": "batting_style",
    "blank2": "bowling_style",
})

# Combine first + surname to match your auction_history player_name format
df_meta_2025["full_name"] = (
    df_meta_2025["first_name"].astype(str) + " " + df_meta_2025["surname"].astype(str)
).str.strip()

pd.set_option("display.max_columns", None)
print(df_meta_2025.shape)
df_meta_2025[["full_name", "country", "dob", "age", "specialism",
              "batting_style", "bowling_style", "c_u_a", "reserve_price_lakh"]].head(10)

(631, 22)


,full_name,country,dob,age,specialism,batting_style,bowling_style,c_u_a,reserve_price_lakh
0,Jos Buttler,England,08/09/1990,34,WICKETKEEPER,RHB,RIGHT ARM Off Spin,Capped,200
1,Shreyas Iyer,India,06/12/1994,30,BATTER,RHB,RIGHT ARM Off Spin,Capped,200
2,Rishabh Pant,India,04/10/1997,27,BATTER,LHB,RIGHT ARM Off Spin,Capped,200
3,Kagiso Rabada,South Africa,25/05/1995,29,BOWLER,RHB,RIGHT ARM Fast,Capped,200
4,Arshdeep Singh,India,05/02/1999,26,BOWLER,LHB,LEFT ARM Fast Medium,Capped,200
5,Mitchell Starc,Australia,30/01/1990,35,BOWLER,LHB,LEFT ARM Fast,Capped,200
6,Yuzvendra Chahal,India,23/07/1990,34,BOWLER,RHB,RIGHT ARM Leg Spin,Capped,200
7,Liam Livingstone,England,04/08/1993,31,ALL-ROUNDER,RHB,RIGHT ARM Leg Spin,Capped,200
8,David Miller,South Africa,10/06/1989,35,BATTER,LHB,NaN,Capped,150
9,KL Rahul,India,18/04/1992,32,WICKETKEEPER,RHB,NaN,Capped,200


In [ ]:
# Filter out any stray NaN entries before comparing
meta_names_clean = {n for n in meta_names if isinstance(n, str)}

# Compare against ONLY the 2025 portion of auction_history (fairer comparison)
auction_2025_names = set(auction_history[auction_history["year"] == 2025]["player_name"])

overlap_2025 = meta_names_clean & auction_2025_names
print(f"Meta names (cleaned): {len(meta_names_clean)}")
print(f"2025 auction_history names: {len(auction_2025_names)}")
print(f"Exact overlap (2025 vs 2025): {len(overlap_2025)}")

sample_unmatched = sorted(meta_names_clean - auction_2025_names)[:30]
print(sample_unmatched)

Meta names (cleaned): 573
2025 auction_history names: 227
Exact overlap (2025 vs 2025): 151
['Aaqib Khan', 'Aarya Desai', 'Abdul Bazith', 'Abhay Choudhary', 'Abhijeet Tomar', 'Abhilash Shetty', 'Abhimanyu Easwaran', 'Abhishek Nair', 'Abhishek Saini', 'Abid Mushtaq', 'Adam Milne', 'Adil Rashid', 'Agni Chopra', 'Ajay Ahlawat', 'Ajitesh Guruswamy', 'Akash Parkar', 'Akash Vashisht', 'Akeal Hosein', 'Akhil Chaudhary', 'Akhil Rawat', 'Akshat Raghuwanshi', 'Alex Carey', 'Ali Khan', 'Alick Athanaze', 'Allah Ghazanfar', 'Alzarri Joseph', 'Aman Khan', 'Amandeep Khare', 'Andre Fletcher', 'Andre Siddarth']


In [ ]:
unsold_2025 = pd.read_csv("../data/clean/unsold_2025.csv")
unsold_names = set(unsold_2025["player_name"])

# Combined: everyone who was in the 2025 auction, sold or unsold
full_2025_pool = auction_2025_names | unsold_names

overlap_full_pool = meta_names_clean & full_2025_pool
print(f"Full 2025 pool (sold + unsold): {len(full_2025_pool)}")
print(f"Meta names matching sold+unsold combined: {len(overlap_full_pool)}")

# Check specifically: do the "unmatched" names show up in unsold_2025?
still_missing = meta_names_clean - full_2025_pool
overlap_check = set(sample_unmatched) & unsold_names
print(f"Of the 30 sample unmatched names, how many are in unsold_2025: {len(overlap_check)}")
print(sorted(still_missing)[:20])

Full 2025 pool (sold + unsold): 329
Meta names matching sold+unsold combined: 242
Of the 30 sample unmatched names, how many are in unsold_2025: 5
['Aaqib Khan', 'Aarya Desai', 'Abdul Bazith', 'Abhay Choudhary', 'Abhijeet Tomar', 'Abhilash Shetty', 'Abhimanyu Easwaran', 'Abhishek Nair', 'Abhishek Saini', 'Abid Mushtaq', 'Agni Chopra', 'Ajay Ahlawat', 'Ajitesh Guruswamy', 'Akash Parkar', 'Akash Vashisht', 'Akhil Chaudhary', 'Akhil Rawat', 'Akshat Raghuwanshi', 'Ali Khan', 'Alick Athanaze']


In [ ]:
# Reload the original 2025 file before we filtered out TBA rows
raw_2025_full = pd.read_csv("../data/raw/auction/auction_2025.csv")
raw_2025_full = raw_2025_full.rename(columns={"Players": "player_name"})

tba_names = set(raw_2025_full[raw_2025_full["Sold"] == "TBA"]["player_name"])

overlap_with_tba = set(sample_unmatched) & tba_names
print(f"TBA player count: {len(tba_names)}")
print(f"Of the 20 sample unmatched names, how many are in TBA: {len(overlap_with_tba)}")

# Full picture: combine sold + unsold + TBA
full_2025_registration = auction_2025_names | unsold_names | tba_names
overlap_all = meta_names_clean & full_2025_registration
print(f"Full registration pool (sold+unsold+TBA): {len(full_2025_registration)}")
print(f"Meta names matching full pool: {len(overlap_all)}")

TBA player count: 292
Of the 20 sample unmatched names, how many are in TBA: 20
Full registration pool (sold+unsold+TBA): 620
Meta names matching full pool: 488


In [ ]:
# Reuse the same normalize_name function from earlier
meta_lookup = defaultdict(list)
for name in meta_names_clean:
    meta_lookup[normalize_name(name)].append(name)

meta_matched = []
meta_ambiguous = []
meta_unmatched = []

for name in full_2025_pool:  # the 329 sold+unsold players
    if name in meta_names_clean:
        meta_matched.append({"pool_name": name, "meta_name": name})
        continue

    key = normalize_name(name)
    candidates = meta_lookup.get(key, [])

    if len(candidates) == 1:
        meta_matched.append({"pool_name": name, "meta_name": candidates[0]})
    elif len(candidates) > 1:
        meta_ambiguous.append({"pool_name": name, "candidates": candidates})
    else:
        meta_unmatched.append({"pool_name": name})

print(f"Matched: {len(meta_matched)}")
print(f"Ambiguous: {len(meta_ambiguous)}")
print(f"No match: {len(meta_unmatched)}")
print()
print("Sample unmatched:", [r["pool_name"] for r in meta_unmatched[:15]])

Matched: 266
Ambiguous: 10
No match: 53

Sample unmatched: ['Rashid Khan', 'Axar Patel', 'Srikar Bharat', 'William O’Rourke', 'Shivam Dube', 'Ramandeep Singh', 'Abishek Porel', 'Dhruv Jurel', 'Mayank Yadav', 'Ravi Bishnoi', 'Khaleel Ahmed', 'Heinrich Klaasen', 'Kumar Kartikeya', 'Shubman Gill', 'Rahul Tewatia']


In [ ]:
# Check: are the unmatched names actually the retained players (base_price = NaN)?
unmatched_names_list = [r["pool_name"] for r in meta_unmatched]

retained_check = df_2025[
    (df_2025["player_name"].isin(unmatched_names_list)) &
    (df_2025["base_price_cr"].isna())
]
print(f"Of {len(unmatched_names_list)} unmatched names, how many have no base price (likely retained): {len(retained_check)}")
print(sorted(retained_check["player_name"].tolist()))

# The genuine leftover mismatches (not retention-related)
non_retention_unmatched = set(unmatched_names_list) - set(retained_check["player_name"])
print(f"\nGenuinely unmatched (not retention): {len(non_retention_unmatched)}")
print(sorted(non_retention_unmatched))

Of 53 unmatched names, how many have no base price (likely retained): 37
['Abishek Porel', 'Andre Russell', 'Axar Patel', 'Ayush Badoni', 'Dhruv Jurel', 'Hardik Pandya', 'Harshit Rana', 'Heinrich Klaasen', 'Jasprit Bumrah', 'MS Dhoni', 'Matheesha Pathirana', 'Mayank Yadav', 'Nicholas Pooran', 'Nitish Kumar Reddy', 'Pat Cummins', 'Rahul Tewatia', 'Rajat Patidar', 'Ramandeep Singh', 'Rashid Khan', 'Ravi Bishnoi', 'Ravindra Jadeja', 'Rinku Singh', 'Riyan Parag', 'Ruturaj Gaikwad', 'Sai Sudharsan', 'Sanju Samson', 'Shimron Hetmyer', 'Shivam Dube', 'Shubman Gill', 'Sunil Narine', 'Tilak Varma', 'Travis Head', 'Tristan Stubbs', 'Varun Chakravarthy', 'Virat Kohli', 'Yash Dayal', 'Yashasvi Jaiswal']

Genuinely unmatched (not retention): 16
['Abdul Basith', 'Aravelly Avanish', 'C Andre Siddarth', 'Jofra Archer', 'Khaleel Ahmed', 'Krishnan Shrijith', 'Kumar Kartikeya', 'Naveen-ul-Haq', 'Rasikh Salam', 'Sai Kishore', 'Satyanarayana Raju', 'Srikar Bharat', 'Swastik Chikara', 'Vijaykumar Vyshak', '

In [ ]:
retained_players_meta = {
    "Abishek Porel": {"country": "India", "specialism": "WICKETKEEPER", "batting_style": "LHB", "bowling_style": None},
    "Andre Russell": {"country": "West Indies", "specialism": "ALL-ROUNDER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "Axar Patel": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "LHB", "bowling_style": "LEFT ARM Orthodox"},
    "Ayush Badoni": {"country": "India", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Dhruv Jurel": {"country": "India", "specialism": "WICKETKEEPER", "batting_style": "RHB", "bowling_style": None},
    "Hardik Pandya": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Medium Fast"},
    "Harshit Rana": {"country": "India", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "Heinrich Klaasen": {"country": "South Africa", "specialism": "WICKETKEEPER", "batting_style": "RHB", "bowling_style": None},
    "Jasprit Bumrah": {"country": "India", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "MS Dhoni": {"country": "India", "specialism": "WICKETKEEPER", "batting_style": "RHB", "bowling_style": None},
    "Matheesha Pathirana": {"country": "Sri Lanka", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "Mayank Yadav": {"country": "India", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "Nicholas Pooran": {"country": "West Indies", "specialism": "WICKETKEEPER", "batting_style": "LHB", "bowling_style": None},
    "Nitish Kumar Reddy": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Medium"},
    "Pat Cummins": {"country": "Australia", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Fast"},
    "Rahul Tewatia": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "LHB", "bowling_style": "RIGHT ARM Leg Spin"},
    "Rajat Patidar": {"country": "India", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Ramandeep Singh": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Medium"},
    "Rashid Khan": {"country": "Afghanistan", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Leg Spin"},
    "Ravi Bishnoi": {"country": "India", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Leg Spin"},
    "Ravindra Jadeja": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "LHB", "bowling_style": "LEFT ARM Orthodox"},
    "Rinku Singh": {"country": "India", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
    "Riyan Parag": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Leg Spin"},
    "Ruturaj Gaikwad": {"country": "India", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Sai Sudharsan": {"country": "India", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
    "Sanju Samson": {"country": "India", "specialism": "WICKETKEEPER", "batting_style": "RHB", "bowling_style": None},
    "Shimron Hetmyer": {"country": "West Indies", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
    "Shivam Dube": {"country": "India", "specialism": "ALL-ROUNDER", "batting_style": "LHB", "bowling_style": "RIGHT ARM Medium"},
    "Shubman Gill": {"country": "India", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Sunil Narine": {"country": "West Indies", "specialism": "ALL-ROUNDER", "batting_style": "LHB", "bowling_style": "RIGHT ARM Off Spin"},
    "Tilak Varma": {"country": "India", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
    "Travis Head": {"country": "Australia", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
    "Tristan Stubbs": {"country": "South Africa", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Varun Chakravarthy": {"country": "India", "specialism": "BOWLER", "batting_style": "RHB", "bowling_style": "RIGHT ARM Mystery Spin"},
    "Virat Kohli": {"country": "India", "specialism": "BATTER", "batting_style": "RHB", "bowling_style": None},
    "Yash Dayal": {"country": "India", "specialism": "BOWLER", "batting_style": "LHB", "bowling_style": "LEFT ARM Medium Fast"},
    "Yashasvi Jaiswal": {"country": "India", "specialism": "BATTER", "batting_style": "LHB", "bowling_style": None},
}
print(len(retained_players_meta))